In [2]:
import requests

url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": 40.4093,
    "longitude": 49.8671,
    "start_date": "2026-01-01",
    "end_date": "2026-06-30",
    "daily": [
        "temperature_2m_mean",
        "precipitation_sum",
        "wind_speed_10m_max"
    ],
    "timezone": "Asia/Baku"
}

response = requests.get(url, params=params, timeout=60)

print("HTTP status:", response.status_code)
print("Sorğu uğurludur:", response.ok)

HTTP status: 200
Sorğu uğurludur: True


In [3]:
import json
from pathlib import Path

# Layihənin əsas qovluğunu tapırıq
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# API məlumatlarının saxlanacağı qovluq
API_DIR = PROJECT_ROOT / "data" / "api"
API_DIR.mkdir(parents=True, exist_ok=True)

# API cavabını JSON formatına çeviririk
weather_data = response.json()

# Faylı saxlayırıq
file_path = API_DIR / "baku_weather_2026_01_06.json"

with open(file_path, "w", encoding="utf-8") as file:
    json.dump(weather_data, file, ensure_ascii=False, indent=4)

# Nəticəni yoxlayırıq
print("Fayl saxlanıldı:", file_path)
print("Gün sayı:", len(weather_data["daily"]["time"]))
print("İlk tarix:", weather_data["daily"]["time"][0])
print("Son tarix:", weather_data["daily"]["time"][-1])

Fayl saxlanıldı: /workspaces/Bank_Analysis/data/api/baku_weather_2026_01_06.json
Gün sayı: 181
İlk tarix: 2026-01-01
Son tarix: 2026-06-30


In [4]:
import pandas as pd
import sys
from sqlalchemy import text

# db.py faylını istifadə etmək üçün
sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))
from db import get_engine

# JSON-dakı gündəlik hava məlumatlarını cədvələ çeviririk
weather_df = pd.DataFrame(weather_data["daily"])

# Hava məlumatlarının aid olduğu şəhər
weather_df["city"] = "Bakı"

print("Hava məlumatlarının ölçüsü:", weather_df.shape)
print(weather_df.head())

# SQL Server-ə qoşuluruq
engine = get_engine()

# Bronze cədvəlinə yükləyirik
weather_df.to_sql(
    name="weather",
    con=engine,
    schema="bronze",
    if_exists="replace",
    index=False
)

# SQL Server-də nəticəni yoxlayırıq
with engine.connect() as conn:
    count = conn.execute(
        text("SELECT COUNT(*) FROM bronze.weather")
    ).scalar()

print("\nbronze.weather:", count, "sətir")

Hava məlumatlarının ölçüsü: (181, 5)
         time  temperature_2m_mean  precipitation_sum  wind_speed_10m_max  \
0  2026-01-01                  4.1                0.0                28.7   
1  2026-01-02                  2.9                5.3                36.6   
2  2026-01-03                  2.4                0.0                22.7   
3  2026-01-04                  2.9                0.0                26.9   
4  2026-01-05                  6.1                0.0                23.2   

   city  
0  Bakı  
1  Bakı  
2  Bakı  
3  Bakı  
4  Bakı  

bronze.weather: 181 sətir
